# Interactive Geospatial Analysis with Folium
Creates a map from the IBM course geospatial dataset. Marker colors represent the recorded class value, and marker popups show the launch site and launch outcome.

In [1]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster
from IPython.display import display, IFrame

DATA_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv"
df = pd.read_csv(DATA_URL)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Shape: (56, 13)
Columns: ['Flight Number', 'Date', 'Time (UTC)', 'Booster Version', 'Launch Site', 'Payload', 'Payload Mass (kg)', 'Orbit', 'Customer', 'Landing Outcome', 'class', 'Lat', 'Long']


,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Landing Outcome,class,Lat,Long
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0.0,LEO,SpaceX,Failure (parachute),0,28.562302,-80.577356
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",0.0,LEO (ISS),NASA (COTS) NRO,Failure (parachute),0,28.562302,-80.577356
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525.0,LEO (ISS),NASA (COTS),No attempt,0,28.562302,-80.577356
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356


In [2]:
required = ["Lat", "Long", "Launch Site", "class"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise KeyError(f"Missing expected columns: {missing}")

center = [df["Lat"].mean(), df["Long"].mean()]
site_map = folium.Map(location=center, zoom_start=4, tiles="OpenStreetMap")
cluster = MarkerCluster().add_to(site_map)

for _, row in df.dropna(subset=["Lat", "Long"]).iterrows():
    color = "green" if int(row["class"]) == 1 else "red"
    popup = (
        f"Launch site: {row['Launch Site']}<br>"
        f"Latitude: {row['Lat']}<br>Longitude: {row['Long']}<br>"
        f"Class: {row['class']}"
    )
    folium.Marker(
        location=[row["Lat"], row["Long"]],
        popup=folium.Popup(popup, max_width=300),
        icon=folium.Icon(color=color, icon="info-sign")
    ).add_to(cluster)

site_map.save("spacex_launch_sites_map.html")
print("Saved interactive map: spacex_launch_sites_map.html")
site_map

Saved interactive map: spacex_launch_sites_map.html


## Proximity analysis
Do not report precise distances to railways, highways or cities unless you have sourced and measured the corresponding landmark coordinates. The course geospatial dataset supports plotting launch records; it does not by itself substantiate all nearby-landmark distance claims.